# AnnDataset Demo: PyTorch Integration with Custom Transforms

This notebook demonstrates how to use `AnnDataset` with custom Transform classes for multiprocessing-safe data loading.


In [ ]:
import numpy as np
import scanpy as sc
import anndata as ad
from anndata.experimental.pytorch import AnnDataset, Transform, Compose
import torch
from torch.utils.data import DataLoader


## Create Sample Data


In [ ]:
# Create sample single-cell data
n_obs = 1000
n_vars = 2000

# Generate random count data
X = np.random.negative_binomial(5, 0.3, size=(n_obs, n_vars)).astype(np.float32)

# Create AnnData object
adata = ad.AnnData(X=X)
adata.obs['cell_type'] = np.random.choice(['TypeA', 'TypeB', 'TypeC'], n_obs)
adata.obs['batch'] = np.random.choice(['Batch1', 'Batch2'], n_obs)
adata.var_names = [f'Gene_{i}' for i in range(n_vars)]

print(f"Created AnnData: {adata}")


## Define Custom Transform Classes

Create custom transforms that are serializable for multiprocessing:


class NormalizeTransform(Transform):
    """Normalize counts to target sum per cell."""
    def __init__(self, target_sum=1e4):
        self.target_sum = target_sum
    
    def __call__(self, x):
        # Ensure positive values
        x = torch.clamp(x, min=0)
        # Normalize to target sum
        row_sum = torch.sum(x, dim=-1, keepdim=True) + 1e-8
        return x * (self.target_sum / row_sum)
    
    def __repr__(self):
        return f"NormalizeTransform(target_sum={self.target_sum})"

class LogTransform(Transform):
    """Apply log1p transformation."""
    def __call__(self, x):
        return torch.log1p(x)
    
    def __repr__(self):
        return "LogTransform()"

# Test the transforms
sample_data = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
normalize = NormalizeTransform(target_sum=10)
log_transform = LogTransform()

print("Original data:", sample_data)
print("After normalization:", normalize(sample_data))
print("After log transform:", log_transform(normalize(sample_data)))


## Create Dataset and Test Multiprocessing


In [ ]:
# Create composed transform
transform = Compose([
    NormalizeTransform(target_sum=1e4),
    LogTransform()
])

print("Composed transform:", transform)

# Create dataset
dataset = AnnDataset(adata, transform=transform)
print(f"Dataset size: {len(dataset)}")

# Test single sample
sample = dataset[0]
print(f"Sample X shape: {sample['X'].shape}")
print(f"Sample X range: {sample['X'].min():.3f} to {sample['X'].max():.3f}")

# Create DataLoader with multiprocessing - this works!
dataloader = DataLoader(dataset, batch_size=32, num_workers=2, shuffle=True)

# Test batch loading
batch = next(iter(dataloader))
print(f"Batch X shape: {batch['X'].shape}")
print("✅ Multiprocessing with custom transforms works!")
